# OxtaMem â€” prova de qualidade end-to-end (T4)

**O modelo responde melhor quando o OxtaMem lhe entrega o fato certo**, numa tarefa
onde os fatos excedem a janela de contexto? Mesmo modelo (treinado em recall
associativo MQAR). No teste: **modelo-so** (janela recente de C fatos) vs
**modelo+OxtaMem** (recupera o fato via `forward_trunk` e injeta).

Robusto: erros de build aparecem inteiros. A validacao oficial exige OxtaMem
nativo; o fallback numpy fica disponivel apenas para diagnostico. Runtime: **T4**. Rode em ordem.

## 1 â€” Helpers + GPU + clone (branch `nsos-gpu-phases12`)

In [ ]:
import os, sys, subprocess, pathlib, glob, shutil, base64
def sh(cmd, cwd=None, extra_env=None, quiet=False):
    """Roda um comando mostrando o ERRO REAL (stdout+stderr) se falhar."""
    env = dict(os.environ);
    if extra_env: env.update(extra_env)
    # string -> roda no shell (necessario p/ pipes como 'curl | sh'); lista -> direto
    r = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True, shell=isinstance(cmd, str))
    if r.returncode != 0:
        shown = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))
        print("COMANDO FALHOU:", shown)
        if r.stdout.strip(): print("--- stdout (fim) ---\n" + r.stdout[-3500:])
        if r.stderr.strip(): print("--- stderr (fim) ---\n" + r.stderr[-3500:])
        raise RuntimeError(f"exit {r.returncode}: {shown}")
    elif not quiet and r.stdout.strip():
        print(r.stdout[-800:])
    return r

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
REPO = '/content/reimagined'; BRANCH = 'nsos-gpu-phases12'
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/vitorGgC569/reimagined.git')
TOKEN = None
try:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'GH_TOKEN'):
        try:
            TOKEN = userdata.get(key)
            if TOKEN: print('token', key, 'ok'); break
        except Exception: pass
except Exception as e:
    print('userdata indisponivel:', e)
if not TOKEN and 'github.com' in REPO_URL:
    import getpass
    TOKEN = getpass.getpass('GitHub token (entrada oculta): ')
assert TOKEN or 'github.com' not in REPO_URL, 'repo privado: configure o Colab Secret GITHUB_TOKEN (icone chave)'
git_env = {}
if TOKEN:
    auth = base64.b64encode(f'oauth2:{TOKEN}'.encode()).decode()
    git_env = {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.extraHeader',
               'GIT_CONFIG_VALUE_0': f'Authorization: Basic {auth}'}
if not pathlib.Path(REPO, '.git').exists():
    if pathlib.Path(REPO).exists(): shutil.rmtree(REPO)
    sh(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO], extra_env=git_env)
else:
    sh(['git', '-C', REPO, 'fetch', '--depth', '1', 'origin', BRANCH], extra_env=git_env)
    sh(['git', '-C', REPO, 'reset', '--hard', 'FETCH_HEAD'])
commit = sh(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'], quiet=True).stdout.strip()
os.environ['REPO'] = REPO
print('repo pronto:', REPO, '| branch', BRANCH, '| commit', commit)

## 2 â€” Build do `nsos_ext` (CUDA â€” receita do bootstrap oficial)

In [ ]:
BUILD = f'{REPO}/OXN/nsos/build-t4'
sh(['apt-get', 'update', '-qq'])
sh(['apt-get', 'install', '-y', '-qq', 'cmake', 'ninja-build', 'build-essential'])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'pybind11>=2.10', 'numpy'])
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout
arch = next((v for k, v in {'T4':'75','V100':'70','L4':'89','A100':'80','H100':'90'}.items() if k in gpu), '75')
print('GPU', gpu.strip(), '-> sm_' + arch)
sh(['cmake', '-G', 'Ninja', '-S', f'{REPO}/OXN/nsos', '-B', BUILD,
    '-DCMAKE_BUILD_TYPE=Release', '-DNSOS_ENABLE_CUDA=ON', '-DNSOS_BUILD_PYTHON=ON',
    '-DNSOS_BUILD_TESTS=OFF', '-DNSOS_BUILD_CLI=OFF', '-DNSOS_BUILD_API=OFF',
    '-DNSOS_BUILD_OXTAMEM=OFF', f'-DNSOS_CUDA_ARCHITECTURES={arch}'])
sh(['cmake', '--build', BUILD, '--parallel', str(os.cpu_count() or 2), '--target', 'nsos_ext'])
so = glob.glob(f'{BUILD}/**/nsos_ext*.so', recursive=True)
assert so, 'nsos_ext nao compilou'
sys.path.insert(0, str(pathlib.Path(so[0]).parent))
import nsos_ext as nsos
print('nsos_ext OK:', so[0])

## 3 â€” Memoria vetorial: OxtaMem nativo (rust+maturin)

A validacao oficial exige o OxtaMem nativo. O store numpy com retrieval cosine
identico permanece apenas como ferramenta diagnostica (`REQUIRE_NATIVE = False`).

In [ ]:
import numpy as np
class NumpyMemStore:
    """Fallback de memoria vetorial: cosine KNN exato. Mesma interface do
    PyGeodesicEngine usada aqui (write_with_vector / search_similar / flush)."""
    def __init__(self): self.vecs=[]; self.values=[]
    def set_sync_on_write(self, b): pass
    def flush(self): pass
    def write_with_vector(self, token_id, value, vector):
        v = np.asarray(vector, dtype=np.float32); v = v/(np.linalg.norm(v)+1e-8)
        self.vecs.append(v); self.values.append(bytes(value)); return len(self.values)-1
    def search_similar(self, vector, k):
        if not self.vecs: return []
        q = np.asarray(vector, dtype=np.float32); q = q/(np.linalg.norm(q)+1e-8)
        sims = np.stack(self.vecs) @ q
        idx = np.argsort(-sims)[:k]
        return [self.values[i] for i in idx]

REQUIRE_NATIVE = True
USE_OXTA = False; PyGeodesicEngine = None
try:
    sh(['apt-get', 'install', '-y', '-qq', 'patchelf'])   # maturin repair dep
    if shutil.which('cargo') is None:
        print('instalando rust toolchain...')
        sh('curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal')
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'maturin'])
    # --compatibility linux evita o repair manylinux (que exige patchelf/docker e quebra no Colab)
    sh(['maturin', 'build', '--release', '--compatibility', 'linux', '--interpreter', sys.executable],
       cwd=f'{REPO}/modules/oxtamem')
    whl = glob.glob(f'{REPO}/modules/oxtamem/oxta_engine/target/wheels/oxta_mem-*.whl')
    assert whl, 'wheel nao gerado'
    sh([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', whl[0]])
    from oxta_mem.native import PyGeodesicEngine
    USE_OXTA = True
    print('>>> OxtaMem NATIVO OK:', whl[0])
except Exception as e:
    print('>>> OxtaMem nativo indisponivel. Motivo:', repr(e)[:400])
    if REQUIRE_NATIVE:
        raise RuntimeError('Validacao oficial exige OxtaMem nativo; fallback numpy nao e prova do OxtaMem') from e
print('memoria em uso:', 'OxtaMem nativo' if USE_OXTA else 'NumpyMemStore (cosine KNN)')

In [ ]:
sh(['cargo', 'test', '--release', '--locked'], cwd=f'{REPO}/modules/oxtamem/oxta_engine')

import gc, tempfile
root_gate = tempfile.mkdtemp(prefix='oxta_gate_', dir='/content')
db_gate = os.path.join(root_gate, 'store.db')
gate = PyGeodesicEngine(db_gate, 8)
gate.set_sync_on_write(False)
vectors_gate = []
for i in range(10):
    v = np.zeros(128, dtype=np.float32); v[i] = 1.0
    vectors_gate.append(v.tolist())
    gate.write_with_vector(str(i), bytes([i]), v.tolist())
gate.flush()
assert all(gate.read_latest(str(i)) == bytes([i]) for i in range(10))
assert all(gate.search_similar(vectors_gate[i], 1)[0] == bytes([i]) for i in range(10))
del gate; gc.collect()

gate = PyGeodesicEngine(db_gate, 8)
assert all(gate.read_latest(str(i)) == bytes([i]) for i in range(10))
assert all(gate.search_similar(vectors_gate[i], 1)[0] == bytes([i]) for i in range(10))
del gate; gc.collect(); shutil.rmtree(root_gate, ignore_errors=True)
print('>>> OxtaMem alignment/persistence gate: 10/10 OK')

## 4 â€” Tarefa de memoria (MQAR) + treino

Modelo aprende lookup associativo em contexto (`k1 v1 ... [Q] kq` -> `vq`), com
comprimento variavel 1..8 -> robusto tanto ao 1 fato injetado quanto a janela cheia.

In [ ]:
import numpy as np, random, math, time
GPU = nsos.Device.GPU
os.environ['NSOS_ATTN_LAST_GUARD'] = '0'   # hibrido [M,A,M,A] (atencao ajuda recall)
QUERY = 1; V0, V1 = 2, 66; K0, K1 = 66, 1266; VOCAB = 1266   # values byte-safe (<256); muitas keys
D_MODEL, D_STATE = 128, 64; N_PAIRS = 8; BATCH = 32; LR = 2e-3; STEPS = 2500; SEED = 0

def _set(c, k, v):
    try: setattr(c, k, v)
    except Exception: pass
def build():
    c = nsos.ModelConfig(); c.num_layers=4; c.d_model=D_MODEL; c.vocab_size=VOCAB
    c.n_heads=4; c.n_kv_heads=2; c.max_context_tokens=1024
    c.use_moe=False; c.use_kan=False; c.use_ttt=False; c.dropout=0.0
    _set(c,'mamba2_faithful',True); _set(c,'mamba_expand',2); _set(c,'mamba_head_dim',64)
    _set(c,'mamba_state_expansion',D_STATE); _set(c,'mamba_n_groups',1)
    c.attention_period=2; c.attention_slot=1
    m = nsos.JambaModel(c, GPU); m.to(GPU); return m

def mqar_seq(rng, n_pairs):
    keys = rng.sample(range(K0, K1), n_pairs)
    vals = [rng.randrange(V0, V1) for _ in range(n_pairs)]
    seq = []
    for k, v in zip(keys, vals): seq += [k, v]
    qi = rng.randrange(n_pairs); seq += [QUERY, keys[qi]]
    return seq, vals[qi]

def batch(rng, n_pairs):
    ps, an = [], []
    for _ in range(BATCH):
        s, v = mqar_seq(rng, n_pairs); ps.append(s); an.append([v])
    return ps, an

nsos.set_seed(SEED); model = build()
tr = nsos.Trainer(model, LR); tr.warmup_steps=100; tr.total_training_steps=STEPS
tr.phase_scheduler.progressive_qat_enabled=False
rng = random.Random(7); model.set_training_mode(True); t0=time.time(); run=0.0
for s in range(STEPS):
    npair = rng.randint(1, N_PAIRS)
    ps, an = batch(rng, npair); run += tr.train_supervised_batch(ps, an)
    if (s+1) % 500 == 0: print(f'  step {s+1}/{STEPS} loss~{run/500:.3f}'); run=0.0
model.set_training_mode(False)
print(f'treino ok em {time.time()-t0:.0f}s')

## 5 â€” Sanity: o modelo aprendeu o lookup em contexto?

In [ ]:
def _argmax_last(ids):
    lg = np.asarray(model.forward_ids(ids).cpu().numpy()).reshape(-1, VOCAB)
    return int(lg[-1].argmax())
vr = random.Random(999); ok = 0; T = 300
for _ in range(T):
    s, v = mqar_seq(vr, N_PAIRS); ok += (_argmax_last(s) == v)
acc = ok / T
print(f'MQAR in-context acc (n_pairs={N_PAIRS}) = {acc:.3f}')
assert acc > 0.6, 'modelo nao aprendeu o lookup â€” aumente STEPS ou cheque o build'

## 6 â€” Prova end-to-end: modelo-so vs modelo+OxtaMem (fatos >> contexto)

In [ ]:
def argmax_last(ids):
    lg = np.asarray(model.forward_ids(ids).cpu().numpy()).reshape(-1, VOCAB)
    return int(lg[-1].argmax())

def embed_key(k):
    tr_ = np.asarray(model.forward_trunk([QUERY, k]).cpu().numpy()).reshape(-1, D_MODEL)
    return tr_[-1].astype(np.float32).tolist()

def make_store():
    if USE_OXTA:
        import tempfile
        root = tempfile.mkdtemp(prefix='oxta_qa_', dir='/content')
        db = os.path.join(root, 'qa_store.db')
        eng = PyGeodesicEngine(db, 8); eng.set_sync_on_write(False); return eng, root
    return NumpyMemStore(), None

def run_eval(N, C=8, n_query=200, seed=0):
    rng = random.Random(seed)
    keys = rng.sample(range(K0, K1), min(N, K1 - K0)); N = len(keys)
    vals = {k: rng.randrange(V0, V1) for k in keys}
    store, store_root = make_store()
    for k in keys:
        store.write_with_vector(f'k{k}', bytes([vals[k]]), embed_key(k))
    store.flush()
    order = keys[:]                       # ordem de chegada; ultimos C = janela do modelo-so
    qrng = random.Random(seed * 7 + 1)
    qs = [qrng.randrange(N) for _ in range(n_query)]
    alone = oxta = retr = 0
    for qi in qs:
        qk = order[qi]; true_v = vals[qk]
        window = order[max(0, N - C):]
        seq = []
        for k in window: seq += [k, vals[k]]
        seq += [QUERY, qk]
        alone += (argmax_last(seq) == true_v)      # modelo-so (so a janela)
        got = store.search_similar(embed_key(qk), 1)
        rv = got[0][0] if got else -1              # value recuperado da memoria
        retr += (rv == true_v)
        oxta += (argmax_last([qk, rv, QUERY, qk]) == true_v)   # modelo + fato injetado
    result = dict(N=N, C=C, alone=alone/len(qs), oxta=oxta/len(qs), retr=retr/len(qs))
    if store_root:
        import gc
        del store; gc.collect(); shutil.rmtree(store_root, ignore_errors=True)
    return result

print(f'store: {"OxtaMem nativo" if USE_OXTA else "NumpyMemStore"}')
print(f'{"N":>6} {"C(janela)":>10} {"modelo-so":>11} {"modelo+OxtaMem":>15} {"(retr@1)":>10}')
results = []
for N in (50, 200, 1000):
    r = run_eval(N, C=8)
    results.append(r)
    print(f'{r["N"]:>6} {r["C"]:>10} {r["alone"]:>11.3f} {r["oxta"]:>15.3f} {r["retr"]:>10.3f}')
assert USE_OXTA, 'validacao oficial exige OxtaMem nativo'
assert min(r['retr'] for r in results) >= 0.99, 'recall exato do OxtaMem abaixo de 99%'
assert results[-1]['oxta'] > results[-1]['alone'] + 0.30, 'OxtaMem nao melhorou o caso N=1000'

## 7 â€” Leitura

- **modelo-so**: so acerta se o fato caiu na janela dos ultimos C -> acc ~ C/N, cai com N.
- **modelo+OxtaMem**: recupera o fato certo da memoria O(N) e injeta -> acc ~ retrieval (~1.0).
- `modelo+OxtaMem >> modelo-so` em N grande = **o OxtaMem faz o modelo responder melhor**
  (qualidade end-to-end, nao so capacidade). O store usado (nativo/numpy) faz o MESMO
  retrieval cosine â€” a conclusao independe da implementacao.